# NanoChat-X — Google Colab Training

Keeps the model architecture at 
 - `block_size=256`, 
 - `n_layer=8`, 
 - `n_head=8`, 
 - `n_embd=512`, while lowering the micro-batch size to reduce CUDA memory use.

**Checkpoint saving:** the training cell watches `out/ckpt.pt` and copies each stable update to Google Drive using a timestamped filename. It also saves a final copy when training exits. This preserves snapshots across Colab runtime resets, provided Google Drive is mounted successfully.

**Before running:** In Colab, select **Runtime → Change runtime type → GPU**. If CUDA is already initialized or a previous run left memory occupied, use **Runtime → Restart session** first. Run cells from top to bottom. When prompted, authorize Google Drive access.

In [ ]:
# Configure CUDA allocator before importing torch (best after a fresh runtime restart)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
gc.collect()

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free: {free / 1024**3:.2f} GiB / {total / 1024**3:.2f} GiB")
    torch.cuda.empty_cache()
else:
    raise RuntimeError("No GPU detected. In Colab, select Runtime → Change runtime type → GPU.")

In [ ]:
# Clone a fresh copy of the repository
%cd /content
!rm -rf NanoChat-X
!git clone https://github.com/mohalkarushikesh/NanoChat-X.git
%cd /content/NanoChat-X
!pip install -r requirements.txt

In [ ]:
# Replace src/config.py with a memory-friendlier default configuration.
# Architecture is preserved; batch_size is reduced from 48 to 4.
from pathlib import Path

config_content = '''"""Central configuration for NanoChat-X."""

from __future__ import annotations

from dataclasses import dataclass, field, asdict, fields
from typing import Any


@dataclass
class GPTConfig:
    """Architecture of the causal transformer (a small GPT)."""

    vocab_size: int = 256
    block_size: int = 256
    n_layer: int = 8
    n_head: int = 8
    n_embd: int = 512
    dropout: float = 0.1
    bias: bool = True

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError(
                f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})"
            )


@dataclass
class TrainConfig:
    """Everything about a training run (model config included)."""

    model: GPTConfig = field(default_factory=GPTConfig)
    data_path: str = "data/data.txt"
    tokenizer: str = "word"
    val_fraction: float = 0.1
    batch_size: int = 4
    grad_accum_steps: int = 1
    max_iters: int = 25000
    learning_rate: float = 5e-4
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    warmup_iters: int = 300
    lr_decay_iters: int = 25000
    min_lr: float = 3e-5
    eval_interval: int = 250
    eval_iters: int = 50
    log_interval: int = 50
    out_dir: str = "out"
    device: str = "auto"
    seed: int = 1337
    compile: bool = False

    def resolved_device(self) -> str:
        if self.device != "auto":
            return self.device
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "TrainConfig":
        d = dict(d)
        model = d.pop("model", {})
        cfg = cls(**d)
        cfg.model = GPTConfig(**model)
        return cfg

    def apply_overrides(self, overrides: dict[str, Any]) -> None:
        """Set fields by name; GPTConfig fields are routed to model."""
        model_fields = {f.name for f in fields(GPTConfig)}
        train_fields = {f.name for f in fields(TrainConfig)}
        for key, value in overrides.items():
            if key in model_fields:
                setattr(self.model, key, _coerce(getattr(self.model, key), value))
            elif key in train_fields and key != "model":
                setattr(self, key, _coerce(getattr(self, key), value))
            else:
                raise KeyError(f"Unknown config field: {key!r}")
        self.model.__post_init__()


def _coerce(current: Any, value: Any) -> Any:
    """Cast CLI strings to the type of the existing default."""
    if isinstance(value, str):
        if isinstance(current, bool):
            return value.lower() in ("1", "true", "yes", "y")
        if isinstance(current, int) and not isinstance(current, bool):
            return int(value)
        if isinstance(current, float):
            return float(value)
    return value
'''

Path('src/config.py').write_text(config_content, encoding='utf-8')
print('Updated src/config.py')
print(Path('src/config.py').read_text()[:700])

In [ ]:
# Preprocess the Cornell dataset
!python -m src.preprocess_cornell

In [ ]:
# Verify the model/training defaults before starting
from src.config import GPTConfig, TrainConfig
cfg = TrainConfig()
print("Model:", cfg.model)
print("Batch size:", cfg.batch_size)
print("Gradient accumulation configured:", cfg.grad_accum_steps)
print("Note: grad_accum_steps is 1 unless src/train.py explicitly implements accumulation.")

In [ ]:
# Train and automatically save timestamped checkpoint snapshots to Google Drive.
# This watches the repository's normal checkpoint (out/ckpt.pt), so src/train.py
# does not need to be rewritten. A new snapshot is saved whenever that file changes.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import os
import shutil
import subprocess
import time

repo_dir = Path("/content/NanoChat-X")
checkpoint = repo_dir / "out" / "ckpt.pt"
backup_dir = Path("/content/drive/MyDrive/NanoChat-X-checkpoints")
backup_dir.mkdir(parents=True, exist_ok=True)

print("Timestamped checkpoints will be saved to:", backup_dir)
print("Starting training...")

process = subprocess.Popen(
    ["python", "-m", "src.train"],
    cwd=str(repo_dir),
    stdout=None,
    stderr=None,
)

last_signature = None
snapshot_count = 0

try:
    while process.poll() is None:
        if checkpoint.exists():
            try:
                stat = checkpoint.stat()
                signature = (stat.st_size, stat.st_mtime_ns)

                # Only copy after the checkpoint file has stopped changing,
                # reducing the chance of capturing a partially written file.
                if signature != last_signature:
                    time.sleep(2)
                    if checkpoint.exists():
                        stable_stat = checkpoint.stat()
                        stable_signature = (stable_stat.st_size, stable_stat.st_mtime_ns)
                        if stable_signature == signature:
                            timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
                            snapshot_count += 1
                            destination = backup_dir / f"ckpt_{timestamp}_{snapshot_count:03d}.pt"
                            shutil.copy2(checkpoint, destination)
                            print(f"Saved checkpoint snapshot: {destination.name} ({destination.stat().st_size:,} bytes)")
                            last_signature = signature
            except FileNotFoundError:
                pass
        time.sleep(3)

    if process.returncode != 0:
        raise RuntimeError(f"Training process exited with code {process.returncode}")

finally:
    # Save one final timestamped copy if a checkpoint exists, even if its
    # modification time did not change during the final polling cycle.
    if checkpoint.exists():
        timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
        final_path = backup_dir / f"ckpt_final_{timestamp}.pt"
        shutil.copy2(checkpoint, final_path)
        print(f"Final checkpoint copy: {final_path}")

print("Training finished.")
print(f"Snapshots created during this run: {snapshot_count}")


## If CUDA OOM happens again

1. Restart the Colab session and rerun from the top.
2. Lower `batch_size` from `4` to `2` in `src/config.py` and rerun training.
3. If needed, lower `block_size` from `256` to `128` (this changes the maximum context length, but preserves the 8-layer/512-dimensional width).
4. Timestamped snapshots are copied to `MyDrive/NanoChat-X-checkpoints` when `out/ckpt.pt` changes. This only works if the training script writes that checkpoint file; if the repository uses a different checkpoint path or filename, update `checkpoint` in the training cell.
5. The watcher copies completed/stable file updates; it does not change how often `src/train.py` itself writes checkpoints.

Important: Your latest notebook deletes and reclones /content/NanoChat-X, so restore your latest checkpoint from Google Drive before running the command:

In [ ]:
from pathlib import Path
import shutil

backup_dir = Path("/content/drive/MyDrive/NanoChat-X-checkpoints")
repo_dir = Path("/content/NanoChat-X")
checkpoints = sorted(backup_dir.glob("ckpt_*.pt"))

if not checkpoints:
    raise FileNotFoundError("No timestamped checkpoints found in Google Drive.")

latest = max(checkpoints, key=lambda p: p.stat().st_mtime)
target = repo_dir / "out" / "ckpt.pt"
target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(latest, target)

print("Restored:", latest.name)

In [ ]:
python -m src.train --resume